# 05 - Optional NOAA Weather Integration

Weather data can improve fire risk prediction, but station coverage may be sparse for Moroccos. This notebook is intentionally optional and should not block the FIRMS + NDVI baseline workflow.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Edit this path to the location where you upload the project folder in Google Drive.
PROJECT_DIR = "/content/drive/MyDrive/fire-risk-project"
PROJECT_DIR = Path(PROJECT_DIR)

WEATHER_RAW_DIR = PROJECT_DIR / "data" / "raw" / "weather"
WEATHER_RAW_DIR.mkdir(parents=True, exist_ok=True)

print("WEATHER_RAW_DIR:", WEATHER_RAW_DIR)


Mounted at /content/drive
WEATHER_RAW_DIR: /content/drive/MyDrive/fire-risk-project/data/raw/weather


In [2]:
import os
from getpass import getpass

import pandas as pd
import requests

NOAA_TOKEN = os.getenv("NOAA_TOKEN") or getpass("Enter NOAA_TOKEN, or press Enter to skip NOAA for now: ")
if NOAA_TOKEN:
    print("NOAA token provided for this runtime.")
else:
    print("No NOAA token provided. Keep this notebook as a placeholder until station configuration is ready.")


Enter NOAA_TOKEN, or press Enter to skip NOAA for now: ··········
NOAA token provided for this runtime.


In [3]:
def fetch_noaa_daily_data(
    token,
    station_id,
    start_date="2023-01-01",
    end_date="2023-12-31",
    dataset_id="GHCND",
    datatype_ids=None,
    limit=1000,
):
    """Placeholder NOAA daily API request helper.

    Configure station_id and datatype_ids before running. This returns an empty
    DataFrame instead of failing when configuration is incomplete.
    """
    if datatype_ids is None:
        datatype_ids = ["TMAX", "TMIN", "PRCP", "AWND"]

    if not token or not station_id:
        print("NOAA token or station_id missing; returning an empty DataFrame.")
        return pd.DataFrame()

    url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
    headers = {"token": token}
    params = {
        "datasetid": dataset_id,
        "stationid": station_id,
        "startdate": start_date,
        "enddate": end_date,
        "datatypeid": datatype_ids,
        "units": "metric",
        "limit": limit,
    }

    response = requests.get(url, headers=headers, params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()
    return pd.DataFrame(payload.get("results", []))


## Candidate Variables

Potential NOAA variables for later integration:

- `TMAX`: daily maximum temperature
- `TMIN`: daily minimum temperature
- `PRCP`: daily precipitation
- `AWND`: average daily wind speed


In [4]:
# TODO: Replace with real station IDs after checking NOAA station availability for Morocco and Western Sahara.
STATION_ID = None

weather_df = fetch_noaa_daily_data(
    token=NOAA_TOKEN,
    station_id=STATION_ID,
    start_date="2023-01-01",
    end_date="2023-12-31",
    datatype_ids=["TMAX", "TMIN", "PRCP", "AWND"],
)

if not weather_df.empty:
    output_path = WEATHER_RAW_DIR / "noaa_weather_2023_raw.csv"
    weather_df.to_csv(output_path, index=False)
    print(f"Saved raw NOAA weather data to {output_path}")
    display(weather_df.head())
else:
    print("No weather data saved. This is expected until NOAA station configuration is added.")


NOAA token or station_id missing; returning an empty DataFrame.
No weather data saved. This is expected until NOAA station configuration is added.


## Future Merge Strategy

Weather will be merged later by date and location, depending on station availability.

- Join each grid cell to the nearest station and merge observations by date.
- Or interpolate/aggregate station observations to regional or grid-level daily values.
- Create rolling features using only past weather values:
  - `temp_mean_7d`
  - `rainfall_sum_7d`
  - `rainfall_sum_30d`
  - `wind_mean_7d`

Do not use future weather values when predicting next-7-day fire risk.


In [5]:
def add_placeholder_weather_features(panel):
    """Example structure for later weather feature engineering.

    TODO: Replace placeholder columns with real station-derived or interpolated
    weather features after NOAA data availability is confirmed.
    """
    panel = panel.sort_values(["grid_id", "date"]).copy()

    # TODO: merge weather by nearest station/date or by interpolated grid/date values.
    # TODO: use shift(1) before rolling if the feature should exclude the current day.
    # panel["temp_mean_7d"] = panel.groupby("grid_id")["temp_mean"].transform(lambda s: s.shift(1).rolling(7).mean())
    # panel["rainfall_sum_7d"] = panel.groupby("grid_id")["rainfall"].transform(lambda s: s.shift(1).rolling(7).sum())
    # panel["rainfall_sum_30d"] = panel.groupby("grid_id")["rainfall"].transform(lambda s: s.shift(1).rolling(30).sum())
    # panel["wind_mean_7d"] = panel.groupby("grid_id")["wind"].transform(lambda s: s.shift(1).rolling(7).mean())

    return panel
